In [8]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone
from google.cloud import storage

class GoogleCloudStorage:
    def __init__(self,bucket_name):
        self.client = storage.Client()
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            print(f"Bucket exists  : {bucket_name}")
        except Exception:
            self.bucket = self.client.create_bucket(bucket_name, location=location)
            print(f"Bucket created : {bucket_name}")
            
    def blob_exists(self, blob_path) -> bool:
        '''check if object exists'''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Upload folder function ----------- ###
    def upload_json(self,blob_path,json_data):
        '''upload json file to bucket'''
        blob   = self.bucket.blob(blob_path)
        
        blob.upload_from_string(
            json.dumps(json_data,ensure_ascii = False),
            content_type = "application/json"
        )
        print(f"uploaded JSON -> gs://{self.bucket.name}/{blob_path}")

    def upload_text(self, blob_path, text_data):
        '''upload text file to bucket'''
        blob   = self.bucket.blob(blob_path)

        blob.upload_from_string(
            text_data,
            content_type = "text/plain"
        )
        print(f"Uploaded text -> gs://{self.bucket.name}/{blob_path}")

    def upload_npy(self, blob_path, array):
        '''upload embedding vector'''
        buffer = io.BytesIO()
        np.save(buffer, array)
        buffer.seek(0)
        
        blob = self.bucket.blob(blob_path)
        blob.upload_from_file(
            buffer,
            content_type = "application/octet-stream"
        )
        print(f"Uploaded NPY -> gs://{self.bucket.name}/{blob_path}")
        
    ### ---------- Read file function ----------- ###
    def read_json(self, blob_path):
        '''read json file'''
        blob   = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, blob_path):
        '''read text file'''
        blob   = self.bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, blob_path):
        '''read .npy (embedding vector) file'''
        blob   = self.bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)
        
    ### ---------- Creation folder function ----------- ###
    def create_folder(self,folder_path):
        '''Creating folder and sub folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blob = self.bucket.blob(folder_path)
        blob.upload_from_string("")
        print(f"Folder created : gs://{self.bucket}/{folder_path}")
        
    ### ---------- Remove function ----------- ###
    def delete_blob(self, blob_path):
        blob   = self.bucket.blob(blob_path)
        if blob.exists():
            blob.delete()
        print(f"Deleted: gs://{self.bucket_name}/{blob_path}")

    def delete_folder(self, folder_path):
        '''Remove nest blob(file) in folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blobs = self.bucket.list_blobs(prefix=folder_path)
        count = 0
        for blob in blobs:
            blob.delete()
            count += 1
    
        print(f"Deleted {count} objects under gs://{self.bucket_name}/{folder_path}")

    # def delete_by_ttl(self, prefix, ttl: timedelta):
    #     '''Remove folder with setting time
    #     timeformat support
    #     timedelta(
    #         days=...,
    #         seconds=...,
    #         microseconds=...,
    #         milliseconds=...,
    #         minutes=...,
    #         hours=...,
    #         weeks=...
    #     )
    #     '''
    #     now    = datetime.now(timezone.utc)
    #     blob   = self.bucket.blob(blob_path)
    #     deleted = 0
    #     for blob in blobs:
    #         if blob.time_created and now - blob.time_created > ttl:
    #             blob.delete()
    #             deleted += 1
    #     print(f"TTL cleanup deleted {deleted} objects under {prefix}")
        
# cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")

In [11]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")
student_id = "stu_p003"
metadata = cgs.read_json(f"{student_id}/metadata/metadata.json")
emb1     = cgs.read_npy(f"{student_id}/embedding/embedding01.npy")
emb2     = cgs.read_npy(f"{student_id}/embedding/embedding02.npy")
emb3     = cgs.read_npy(f"{student_id}/embedding/embedding03.npy")
emb4     = cgs.read_npy(f"{student_id}/embedding/embedding04.npy")
emb5     = cgs.read_npy(f"{student_id}/embedding/embedding05.npy")
metadata

Bucket exists  : hyde-datalake-feeds


{'student_id': 'stu_p003',
 'current_status': 'student4+yr',
 'education_level': 'bachelor',
 'education_major': 'สถิติ',
 'target_roles': 'Data Analyst',
 'timezone': 'UTC',
 'model_name': 'gemini-2.5-flash',
 'max_output_tokens': 2048,
 'feed_text_max_chars': 240,
 'temperature': 0.2}

In [14]:
import requests


In [13]:
import requests

student_id = "stu_p003"
url = f"https://hyderecomment-service-v1-1-0-du7yhkyaqq-as.a.run.app/hyde/students/{student_id}/feed"

resp = requests.post(url)
resp.raise_for_status()   # raise error if not 200
data = resp.json()
data

{'student_id': 'stu_p003',
 'metadata': {'student_id': 'stu_p003',
  'current_status': 'student4+yr',
  'education_level': 'bachelor',
  'education_major': 'สถิติ',
  'target_roles': 'Data Analyst',
  'timezone': 'UTC',
  'model_name': 'gemini-2.5-flash',
  'max_output_tokens': 2048,
  'feed_text_max_chars': 240,
  'temperature': 0.2},
 'embedded_vector': {'emb1': [0.0003576562739908695,
   -0.022036602720618248,
   -0.009610642679035664,
   -0.12281759083271027,
   -0.02179572731256485,
   -0.0035793110728263855,
   0.007461403496563435,
   -0.009664880111813545,
   -0.007970403879880905,
   -0.003823535982519388,
   -0.0413154698908329,
   0.0031268857419490814,
   0.0020775229204446077,
   -0.018787724897265434,
   0.17629097402095795,
   -0.009390374645590782,
   0.025073526427149773,
   0.01999911665916443,
   0.017867092043161392,
   -0.02090797945857048,
   0.024305246770381927,
   0.018039599061012268,
   -0.020332608371973038,
   -0.02489609830081463,
   0.016061129048466682,
